In [1]:
"""Zwei Untersuchungen aus dem Bericht (Abschnitte 2.4 und 2.8), nachvollziehbar als Skript.
Ausführen: uv run python src/explore_promo_closures.py   (Ausgabe auch in reports/explore_promo_closures_output.txt)
Vorher: src/clean.py muss gelaufen sein (liest data/clean/train_clean.parquet).
"""

'Zwei Untersuchungen aus dem Bericht (Abschnitte 2.4 und 2.8), nachvollziehbar als Skript.\nAusführen: uv run python src/explore_promo_closures.py   (Ausgabe auch in reports/explore_promo_closures_output.txt)\nVorher: src/clean.py muss gelaufen sein (liest data/clean/train_clean.parquet).\n'

In [2]:
from pathlib import Path

import pandas as pd

ROOT = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd().resolve().parent  # Skript oder Notebook
pd.set_option("display.width", 170)
lines = []


def out(msg=""):
    print(msg)
    lines.append(str(msg))


raw = pd.read_csv(ROOT / "data/raw/train.csv", parse_dates=["date"])

## A. Wann beginnt die Promotion?
Gibt es ein einheitliches Startdatum, oder startet jede Filiale anders? Das entscheidet, wie wir `promo_known` definieren (Bericht 2.4).

In [3]:
out("=== A. Wann beginnt die Promotion? (Bericht 2.4) ===")
out("Frage: gibt es ein einheitliches Startdatum, oder startet jede Filiale anders?")
promo = raw[raw.onpromotion > 0]
first_store = promo.groupby("store_nbr").date.min()
first_fam = promo.groupby("family").date.min()
out("\nErste Aktion je Filiale (Datum: Anzahl Filialen):")
out(first_store.value_counts().sort_index().to_string())
out(f"Filialen ohne je eine Aktion: {raw.store_nbr.nunique() - len(first_store)}")
out("\nErste Aktion je Familie (Datum: Anzahl Familien):")
out(first_fam.value_counts().sort_index().to_string())
out(f"Familien ohne je eine Aktion: {sorted(set(raw.family) - set(first_fam.index))}")
out("\nSpäte Starter (erste Aktion nach 2014-12-31) im Vergleich zur Filialeröffnung:")
daily = raw.groupby(["store_nbr", "date"]).sales.sum()
opening = daily[daily > 0].reset_index().groupby("store_nbr").date.min()
late = first_store[first_store > "2014-12-31"]
out(pd.DataFrame({"erste_Aktion": late, "Eröffnung": opening[late.index]}).to_string())

=== A. Wann beginnt die Promotion? (Bericht 2.4) ===
Frage: gibt es ein einheitliches Startdatum, oder startet jede Filiale anders?

Erste Aktion je Filiale (Datum: Anzahl Filialen):
date
2014-04-01    1
2014-04-15    3
2014-04-29    1
2014-05-12    1
2014-05-13    1
2014-05-14    3
2014-05-15    1
2014-05-19    3
2014-05-20    3
2014-05-21    7
2014-06-11    3
2014-06-12    2
2014-06-13    1
2014-06-16    1
2014-06-17    2
2014-06-19    1
2014-06-25    3
2014-06-26    2
2014-06-27    1
2014-06-30    5
2014-07-01    1
2014-07-12    1
2014-07-25    1
2015-02-13    1
2015-03-20    1
2015-07-24    1
2015-08-21    1
2015-10-09    1
2017-04-21    1
Filialen ohne je eine Aktion: 0

Erste Aktion je Familie (Datum: Anzahl Familien):
date
2014-04-01    2
2014-04-02    1
2014-04-03    5
2014-04-04    1
2014-04-05    1
2014-04-10    2
2014-04-17    2
2014-04-21    1
2014-04-30    1
2014-05-06    1
2014-05-08    1
2014-05-21    1
2014-05-31    1
2014-06-02    1
2014-06-03    1
2014-07-01    4
2014

Familien ohne je eine Aktion: ['BOOKS']

Späte Starter (erste Aktion nach 2014-12-31) im Vergleich zur Filialeröffnung:
          erste_Aktion  Eröffnung
store_nbr                        
20          2015-02-13 2015-02-13
21          2015-07-24 2015-07-24
22          2015-10-09 2015-10-09
29          2015-03-20 2015-03-20
42          2015-08-21 2015-08-21
52          2017-04-21 2017-04-20


## B. Filialen zeitweise geschlossen
Tage mit Gesamtumsatz 0 **nach** der Eröffnung: Einzeltage oder längere Blöcke? Diese Nullen sind kein Nachfragesignal (Bericht 2.8).

In [4]:
out("\n=== B. Filialen zeitweise geschlossen (Bericht 2.8) ===")
out("Frage: Tage mit Gesamtumsatz 0 NACH der Eröffnung. Einzeltage oder längere Blöcke?")
t = pd.read_parquet(ROOT / "data/clean/train_clean.parquet")
c = (t[t.is_store_closed == 1].groupby(["store_nbr", "date"]).size().reset_index()[["store_nbr", "date"]]
     .sort_values(["store_nbr", "date"]))
out(f"Filial-Tage insgesamt: {len(c)}")
out(f"davon am 1. Januar: {(c.date.dt.strftime('%m-%d') == '01-01').sum()}")
out("\nFilial-Tage je Filiale (Top 8):")
out(c.store_nbr.value_counts().head(8).to_string())
c["block"] = (c.groupby("store_nbr").date.diff().dt.days != 1).cumsum()
blocks = c.groupby(["store_nbr", "block"]).date.agg(["min", "max", "count"]).sort_values("count", ascending=False)
out("\nLängste zusammenhängende Schließungen:")
out(blocks.head(8).to_string())
out(f"Blöcke mit mindestens 7 Tagen: {(blocks['count'] >= 7).sum()}   Einzeltage: {(blocks['count'] == 1).sum()}")


=== B. Filialen zeitweise geschlossen (Bericht 2.8) ===
Frage: Tage mit Gesamtumsatz 0 NACH der Eröffnung. Einzeltage oder längere Blöcke?
Filial-Tage insgesamt: 601
davon am 1. Januar: 196

Filial-Tage je Filiale (Top 8):
store_nbr
18    114
24    105
25     66
12     64
14     42
30     27
43      8
17      6

Längste zusammenhängende Schließungen:
                       min        max  count
store_nbr block                             
18        76    2016-08-15 2016-12-02    110
24        93    2014-04-14 2014-07-23    101
25        97    2016-08-22 2016-10-26     66
12        48    2015-03-30 2015-05-28     60
14        56    2014-08-04 2014-09-10     38
30        112   2013-07-08 2013-07-30     23
43        165   2016-11-14 2016-11-17      4
17        71    2016-03-22 2016-03-23      2
Blöcke mit mindestens 7 Tagen: 6   Einzeltage: 197


## C. Feiertagstypen
Welche Typen gibt es, und warum braucht die Tabelle eine Regel (verlegte Feiertage, mehrere Einträge je Tag)? (Bericht 2.1)

In [5]:
out("\n=== C. Feiertagstypen (Bericht 2.1) ===")
hol = pd.read_csv(ROOT / "data/raw/holidays_events.csv", parse_dates=["date"])
out(hol.groupby(["type", "locale"]).size().unstack(fill_value=0).to_string())
out("\nBeispiel: mehrere Einträge am selben Tag und Ort")
out(hol[hol.duplicated(["date", "locale_name"], keep=False)].sort_values("date").head(6).to_string())

(ROOT / "reports" / "explore_promo_closures_output.txt").write_text("\n".join(lines), encoding="utf-8")


=== C. Feiertagstypen (Bericht 2.1) ===
locale      Local  National  Regional
type                                 
Additional     11        40         0
Bridge          0         5         0
Event           0        56         0
Holiday       137        60        24
Transfer        4         8         0
Work Day        0         5         0

Beispiel: mehrere Einträge am selben Tag und Ort
          date        type    locale locale_name                description  transferred
35  2012-12-24      Bridge  National     Ecuador             Puente Navidad        False
36  2012-12-24  Additional  National     Ecuador                  Navidad-1        False
39  2012-12-31      Bridge  National     Ecuador  Puente Primer dia del ano        False
40  2012-12-31  Additional  National     Ecuador       Primer dia del ano-1        False
156 2014-12-26      Bridge  National     Ecuador             Puente Navidad        False
157 2014-12-26  Additional  National     Ecuador                  Navid

3340